# Train Thermal UAV Detector

Notebook-first workflow for local machines, Colab, or SCC Jupyter sessions. It uses the active kernel environment and does not reinstall packages unless you explicitly choose to.

In [ ]:
from pathlib import Path
import os
import sys

try:
    from src.utils.project import find_project_root
except ModuleNotFoundError:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'src').exists() and (candidate / 'configs').exists():
            sys.path.insert(0, str(candidate))
            break
    from src.utils.project import find_project_root

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Python:', sys.executable)

In [ ]:
import torch

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA visible devices:', os.environ.get('CUDA_VISIBLE_DEVICES'))
print('Device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU 0:', torch.cuda.get_device_name(0))
    _probe = torch.zeros(1, device='cuda')
    print('CUDA probe tensor:', _probe)

In [ ]:
# Leave this False on SCC if the kernel already uses your prepared env.
INSTALL_REQUIREMENTS = False

if INSTALL_REQUIREMENTS:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / 'requirements.txt')])

In [ ]:
import subprocess

subprocess.check_call([
    sys.executable,
    '-m', 'src.utils.dataset_checks',
    '--data', 'configs/dataset_thermal_uav.yaml'
], cwd=PROJECT_ROOT)

In [ ]:
from src.detection.train import load_yaml, run_training

TRAIN_CFG = load_yaml(PROJECT_ROOT / 'configs' / 'train_detector.yaml')

# Start with a smoke test on SCC Jupyter, then increase back to the config defaults.
TRAIN_CFG.update({
    'device': 0,
    'epochs': 20,
    'imgsz': 640,
    'batch': -1,
    'workers': 32,
    'cache': 'disk',
    'amp': True,
    'verbose': True,
    'name': 'yolo12n_thermal_uav_speedtest',
})

TRAIN_CFG

In [ ]:
save_dir = run_training(TRAIN_CFG, project_root=PROJECT_ROOT)
save_dir

In [ ]:
from src.detection.validate import load_yaml as load_val_yaml, run_validation

VAL_CFG = {
    'weights': str(save_dir / 'weights' / 'best.pt'),
    'data': 'configs/dataset_thermal_uav.yaml',
    'split': 'val',
    'imgsz': TRAIN_CFG['imgsz'],
    'batch': TRAIN_CFG['batch'],
    'device': 'auto',
    'workers': TRAIN_CFG['workers'],
    'project': 'runs/val',
    'name': f"{TRAIN_CFG['name']}_val",
    'exist_ok': True,
}

metrics_file = run_validation(VAL_CFG, project_root=PROJECT_ROOT)
metrics_file

For the full run, change the training cell overrides to something closer to the config defaults, for example `epochs=100`, `imgsz=960`, and `batch=16` if the notebook kernel has a stable GPU allocation.

In [ ]:
export PYTHONNOUSERSITE=1
python -m src.detection.infer \
  --weights runs/detect/yolo12n_thermal_uav_speedtest3/weights/best.pt \
  --source /projectnb/cs585/students/endrit01/VisionSentry/1.mp4 \
  --imgsz 640 \
  --conf 0.10 \
  --iou 0.5 \
  --device cpu \
  --save_video true \
  --save_frames true \
  --save_txt true \
  --project runs/predict \
  --name test_1mp4_detect_cpu


In [ ]:
export PYTHONNOUSERSITE=1
python -m src.tracking.run_tracker \
  --weights runs/detect/yolo12n_thermal_uav_speedtest3/weights/best.pt \
  --source /projectnb/cs585/students/endrit01/VisionSentry/1.mp4 \
  --tracker configs/tracker_botsort.yaml \
  --imgsz 640 \
  --conf 0.10 \
  --iou 0.5 \
  --device cpu \
  --save_video true \
  --save_frames true \
  --project runs/track \
  --name test_1mp4_track_cpu
